In [1]:
! pip install transformers==4.35.2

In [2]:
import json
import math
import random
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from torch.utils.data import DataLoader, IterableDataset
from transformers import AutoModelForCausalLM, AutoTokenizer, PreTrainedModel, PreTrainedTokenizerBase


PastKeyValues = Tuple[Tuple[torch.Tensor, torch.Tensor], ...]


@dataclass
class TrainConfig:
    model_a_id: str = "gpt2"
    model_b_id: str = "gpt2-medium"
    output_dir: str = "./outputs/lsc_toy"
    max_steps: int = 1000
    batch_size: int = 1
    grad_accum_steps: int = 8
    total_tokens: int = 96
    prefix_tokens: int = 32
    learning_rate: float = 3e-4
    weight_decay: float = 1e-2
    warmup_steps: int = 100
    grad_clip_norm: float = 1.0
    log_every: int = 10
    save_every: int = 100
    seed: int = 42
    shuffle_buffer: int = 10_000
    shared_slots: int = 16
    shared_dim: int = 128
    translator_dim: int = 256
    translator_heads: int = 8
    translator_mlp_ratio: int = 2
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    dtype: str = "float32"


@dataclass
class InferenceConfig:
    checkpoint_path: str = "./outputs/lsc_toy/final_checkpoint.pt"
    max_new_tokens: int = 32
    do_sample: bool = False
    temperature: float = 1.0
    top_k: int = 50
    seed: int = 42
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    prefixes: Tuple[str, str] = (
        "Seoul is the capital of",
        "Paris is the capital of",
    )


@dataclass
class ModelSpec:
    model_id: str
    num_layers: int
    hidden_size: int
    num_heads: int
    head_dim: int


@dataclass
class SharedCache:
    key: torch.Tensor
    value: torch.Tensor


def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_torch_dtype(dtype_name: str) -> torch.dtype:
    mapping = {
        "float32": torch.float32,
        "float": torch.float32,
        "fp32": torch.float32,
        "float16": torch.float16,
        "fp16": torch.float16,
        "bfloat16": torch.bfloat16,
        "bf16": torch.bfloat16,
    }
    key = dtype_name.lower()
    if key not in mapping:
        raise ValueError(f"Unsupported dtype: {dtype_name}")
    return mapping[key]


def load_tokenizer(model_id: str) -> PreTrainedTokenizerBase:
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer


def freeze_model(model: PreTrainedModel) -> None:
    model.eval()
    for param in model.parameters():
        param.requires_grad_(False)


def load_frozen_model(model_id: str, device: str, dtype: str = "float32") -> PreTrainedModel:
    torch_dtype = get_torch_dtype(dtype)
    model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch_dtype)
    model.to(device)
    freeze_model(model)
    return model


def get_model_spec(model: PreTrainedModel) -> ModelSpec:
    config = model.config
    num_heads = getattr(config, "n_head", None)
    hidden_size = getattr(config, "n_embd", None)
    num_layers = getattr(config, "n_layer", None)
    if num_heads is None or hidden_size is None or num_layers is None:
        raise ValueError("This toy example expects GPT-2 style configs with n_head/n_embd/n_layer.")
    if hidden_size % num_heads != 0:
        raise ValueError("hidden_size must be divisible by num_heads.")
    return ModelSpec(
        model_id=getattr(config, "_name_or_path", "unknown"),
        num_layers=num_layers,
        hidden_size=hidden_size,
        num_heads=num_heads,
        head_dim=hidden_size // num_heads,
    )


class OpenWebTextSequenceStream(IterableDataset):
    """
    Streams OpenWebText and yields fixed-length token chunks.

    This behaves like a rolling token buffer, which is usually a better fit for
    language-model style toy experiments than per-document truncation.
    """

    def __init__(
        self,
        tokenizer: PreTrainedTokenizerBase,
        sequence_length: int,
        shuffle_buffer: int = 10_000,
        seed: int = 42,
    ) -> None:
        super().__init__()
        self.tokenizer = tokenizer
        self.sequence_length = sequence_length
        self.shuffle_buffer = shuffle_buffer
        self.seed = seed

    def __iter__(self) -> Iterable[torch.Tensor]:
        stream = load_dataset("openwebtext", split="train", streaming=True)
        stream = stream.shuffle(seed=self.seed, buffer_size=self.shuffle_buffer)
        token_buffer: List[int] = []
        eos_token_id = self.tokenizer.eos_token_id
        for example in stream:
            text = example.get("text", "")
            if not text or text.isspace():
                continue
            token_ids = self.tokenizer(text, add_special_tokens=False).input_ids
            if len(token_ids) < 8:
                continue
            token_buffer.extend(token_ids)
            token_buffer.append(eos_token_id)
            while len(token_buffer) >= self.sequence_length:
                chunk = token_buffer[: self.sequence_length]
                token_buffer = token_buffer[self.sequence_length :]
                yield torch.tensor(chunk, dtype=torch.long)


class CrossAttentionBlock(nn.Module):
    def __init__(self, dim: int, num_heads: int, mlp_ratio: int = 2) -> None:
        super().__init__()
        self.query_norm = nn.LayerNorm(dim)
        self.context_norm = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(embed_dim=dim, num_heads=num_heads, batch_first=True)
        self.ffn_norm = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.Linear(dim, dim * mlp_ratio),
            nn.GELU(),
            nn.Linear(dim * mlp_ratio, dim),
        )

    def forward(self, hidden: torch.Tensor, context: torch.Tensor) -> torch.Tensor:
        q = self.query_norm(hidden)
        kv = self.context_norm(context)
        attn_out, _ = self.attn(q, kv, kv, need_weights=False)
        hidden = hidden + attn_out
        hidden = hidden + self.ffn(self.ffn_norm(hidden))
        return hidden


class LocalToSharedTranslator(nn.Module):
    """
    Toy version of the paper's translator that keeps the spirit of:
      - per-layer preprocessing
      - a stack of cross-attention steps over each source layer
      - a final projection into a shared latent space

    Input:  [batch, seq, local_layers, local_hidden]
    Output: [batch, seq, shared_slots, shared_dim]
    """

    def __init__(
        self,
        local_hidden_size: int,
        local_layers: int,
        shared_slots: int,
        shared_dim: int,
        translator_dim: int,
        translator_heads: int,
        mlp_ratio: int,
    ) -> None:
        super().__init__()
        self.local_layers = local_layers
        self.shared_slots = shared_slots
        self.shared_dim = shared_dim
        self.input_norm = nn.LayerNorm(local_hidden_size)
        self.input_proj = nn.Linear(local_hidden_size, translator_dim)
        self.blocks = nn.ModuleList(
            [
                CrossAttentionBlock(
                    dim=translator_dim,
                    num_heads=translator_heads,
                    mlp_ratio=mlp_ratio,
                )
                for _ in range(local_layers)
            ]
        )
        self.output_norm = nn.LayerNorm(local_layers * translator_dim)
        self.output_proj = nn.Linear(local_layers * translator_dim, shared_slots * shared_dim)

    def forward(self, local_block: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len, _, _ = local_block.shape
        projected = F.gelu(self.input_proj(self.input_norm(local_block)))
        hidden = projected[:, :, 0, :]
        collected = []
        for layer_idx, block in enumerate(self.blocks):
            hidden = block(hidden, projected[:, :, layer_idx, :])
            collected.append(hidden)
        fused = torch.cat(collected, dim=-1)
        shared = F.gelu(self.output_proj(self.output_norm(fused)))
        return shared.view(batch_size, seq_len, self.shared_slots, self.shared_dim)


class SharedToLocalTranslator(nn.Module):
    """
    Reverse map from the shared latent space back into a model-specific KV block.

    Input:  [batch, seq, shared_slots, shared_dim]
    Output: [batch, seq, local_layers, local_hidden]
    """

    def __init__(
        self,
        local_hidden_size: int,
        local_layers: int,
        shared_slots: int,
        shared_dim: int,
        translator_dim: int,
        translator_heads: int,
        mlp_ratio: int,
    ) -> None:
        super().__init__()
        self.local_layers = local_layers
        self.local_hidden_size = local_hidden_size
        self.shared_slots = shared_slots
        self.shared_dim = shared_dim
        flat_shared = shared_slots * shared_dim
        self.input_norm = nn.LayerNorm(flat_shared)
        self.input_proj = nn.Linear(flat_shared, local_layers * translator_dim)
        self.blocks = nn.ModuleList(
            [
                CrossAttentionBlock(
                    dim=translator_dim,
                    num_heads=translator_heads,
                    mlp_ratio=mlp_ratio,
                )
                for _ in range(local_layers)
            ]
        )
        self.output_norm = nn.LayerNorm(local_layers * translator_dim)
        self.output_proj = nn.Linear(local_layers * translator_dim, local_layers * local_hidden_size)

    def forward(self, shared_block: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len, _, _ = shared_block.shape
        flat_shared = shared_block.reshape(batch_size, seq_len, self.shared_slots * self.shared_dim)
        expanded = F.gelu(self.input_proj(self.input_norm(flat_shared)))
        expanded = expanded.view(batch_size, seq_len, self.local_layers, -1)
        hidden = expanded[:, :, 0, :]
        collected = []
        for layer_idx, block in enumerate(self.blocks):
            hidden = block(hidden, expanded[:, :, layer_idx, :])
            collected.append(hidden)
        fused = torch.cat(collected, dim=-1)
        local = F.gelu(self.output_proj(self.output_norm(fused)))
        return local.view(batch_size, seq_len, self.local_layers, self.local_hidden_size)


class ModelLatentAdapter(nn.Module):
    def __init__(
        self,
        model_name: str,
        local_layers: int,
        local_hidden_size: int,
        shared_slots: int,
        shared_dim: int,
        translator_dim: int,
        translator_heads: int,
        mlp_ratio: int,
    ) -> None:
        super().__init__()
        self.model_name = model_name
        self.key_to_shared = LocalToSharedTranslator(
            local_hidden_size=local_hidden_size,
            local_layers=local_layers,
            shared_slots=shared_slots,
            shared_dim=shared_dim,
            translator_dim=translator_dim,
            translator_heads=translator_heads,
            mlp_ratio=mlp_ratio,
        )
        self.value_to_shared = LocalToSharedTranslator(
            local_hidden_size=local_hidden_size,
            local_layers=local_layers,
            shared_slots=shared_slots,
            shared_dim=shared_dim,
            translator_dim=translator_dim,
            translator_heads=translator_heads,
            mlp_ratio=mlp_ratio,
        )
        self.key_from_shared = SharedToLocalTranslator(
            local_hidden_size=local_hidden_size,
            local_layers=local_layers,
            shared_slots=shared_slots,
            shared_dim=shared_dim,
            translator_dim=translator_dim,
            translator_heads=translator_heads,
            mlp_ratio=mlp_ratio,
        )
        self.value_from_shared = SharedToLocalTranslator(
            local_hidden_size=local_hidden_size,
            local_layers=local_layers,
            shared_slots=shared_slots,
            shared_dim=shared_dim,
            translator_dim=translator_dim,
            translator_heads=translator_heads,
            mlp_ratio=mlp_ratio,
        )

    def to_shared(self, key_block: torch.Tensor, value_block: torch.Tensor) -> SharedCache:
        return SharedCache(
            key=self.key_to_shared(key_block),
            value=self.value_to_shared(value_block),
        )

    def from_shared(self, shared_cache: SharedCache) -> Tuple[torch.Tensor, torch.Tensor]:
        key_block = self.key_from_shared(shared_cache.key)
        value_block = self.value_from_shared(shared_cache.value)
        return key_block, value_block


class SharedKVTranslatorPool(nn.Module):
    def __init__(
        self,
        model_specs: Dict[str, ModelSpec],
        shared_slots: int,
        shared_dim: int,
        translator_dim: int,
        translator_heads: int,
        mlp_ratio: int,
    ) -> None:
        super().__init__()
        self.model_specs = model_specs
        self.adapters = nn.ModuleDict(
            {
                name: ModelLatentAdapter(
                    model_name=name,
                    local_layers=spec.num_layers,
                    local_hidden_size=spec.hidden_size,
                    shared_slots=shared_slots,
                    shared_dim=shared_dim,
                    translator_dim=translator_dim,
                    translator_heads=translator_heads,
                    mlp_ratio=mlp_ratio,
                )
                for name, spec in model_specs.items()
            }
        )

    def translate_blocks(
        self,
        key_block: torch.Tensor,
        value_block: torch.Tensor,
        src_name: str,
        dst_name: str,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        shared_cache = self.adapters[src_name].to_shared(key_block, value_block)
        return self.adapters[dst_name].from_shared(shared_cache)

    def translate_past_key_values(
        self,
        past_key_values: PastKeyValues,
        src_name: str,
        dst_name: str,
        dst_spec: ModelSpec,
    ) -> PastKeyValues:
        key_block, value_block = past_key_values_to_blocks(past_key_values)
        translated_key, translated_value = self.translate_blocks(key_block, value_block, src_name=src_name, dst_name=dst_name)
        return blocks_to_past_key_values(
            key_block=translated_key,
            value_block=translated_value,
            model_spec=dst_spec,
        )


@torch.no_grad()
def extract_past_key_values(model: PreTrainedModel, input_ids: torch.Tensor) -> PastKeyValues:
    outputs = model(input_ids=input_ids, use_cache=True)
    return outputs.past_key_values


def past_key_values_to_blocks(past_key_values: PastKeyValues) -> Tuple[torch.Tensor, torch.Tensor]:
    key_layers = []
    value_layers = []
    for key, value in past_key_values:
        batch_size, num_heads, seq_len, head_dim = key.shape
        key_flat = key.permute(0, 2, 1, 3).contiguous().view(batch_size, seq_len, num_heads * head_dim)
        value_flat = value.permute(0, 2, 1, 3).contiguous().view(batch_size, seq_len, num_heads * head_dim)
        key_layers.append(key_flat)
        value_layers.append(value_flat)
    key_block = torch.stack(key_layers, dim=2)
    value_block = torch.stack(value_layers, dim=2)
    return key_block, value_block


def blocks_to_past_key_values(
    key_block: torch.Tensor,
    value_block: torch.Tensor,
    model_spec: ModelSpec,
) -> PastKeyValues:
    batch_size, seq_len, num_layers, hidden_size = key_block.shape
    if num_layers != model_spec.num_layers:
        raise ValueError(f"Layer mismatch: block has {num_layers}, model expects {model_spec.num_layers}.")
    if hidden_size != model_spec.hidden_size:
        raise ValueError(f"Hidden mismatch: block has {hidden_size}, model expects {model_spec.hidden_size}.")

    past_key_values = []
    for layer_idx in range(model_spec.num_layers):
        key_layer = key_block[:, :, layer_idx, :]
        value_layer = value_block[:, :, layer_idx, :]
        key_layer = key_layer.view(batch_size, seq_len, model_spec.num_heads, model_spec.head_dim)
        value_layer = value_layer.view(batch_size, seq_len, model_spec.num_heads, model_spec.head_dim)
        key_layer = key_layer.permute(0, 2, 1, 3).contiguous()
        value_layer = value_layer.permute(0, 2, 1, 3).contiguous()
        past_key_values.append((key_layer, value_layer))
    return tuple(past_key_values)


def flatten_past_key_values(past_key_values: PastKeyValues) -> torch.Tensor:
    flat_parts = []
    for key, value in past_key_values:
        flat_parts.append(key.reshape(key.shape[0], -1))
        flat_parts.append(value.reshape(value.shape[0], -1))
    return torch.cat(flat_parts, dim=1)


def cosine_similarity_between_past(a: PastKeyValues, b: PastKeyValues) -> float:
    flat_a = flatten_past_key_values(a)
    flat_b = flatten_past_key_values(b)
    return F.cosine_similarity(flat_a, flat_b, dim=1).mean().item()


def count_trainable_parameters(module: nn.Module) -> int:
    return sum(param.numel() for param in module.parameters() if param.requires_grad)


def move_past_to_device(past_key_values: PastKeyValues, device: str) -> PastKeyValues:
    moved = []
    for key, value in past_key_values:
        moved.append((key.to(device), value.to(device)))
    return tuple(moved)


def compute_suffix_lm_loss(
    target_model: PreTrainedModel,
    translated_past_key_values: PastKeyValues,
    lm_input_ids: torch.Tensor,
    lm_labels: torch.Tensor,
) -> torch.Tensor:
    outputs = target_model(
        input_ids=lm_input_ids,
        past_key_values=translated_past_key_values,
        use_cache=False,
    )
    logits = outputs.logits
    vocab_size = logits.shape[-1]
    loss = F.cross_entropy(
        logits.reshape(-1, vocab_size),
        lm_labels.reshape(-1),
        reduction="mean",
    )
    return loss


class InfiniteDataLoader:
    def __init__(self, dataloader: DataLoader) -> None:
        self.dataloader = dataloader
        self.iterator = iter(self.dataloader)

    def __iter__(self):
        return self

    def __next__(self):
        try:
            return next(self.iterator)
        except StopIteration:
            self.iterator = iter(self.dataloader)
            return next(self.iterator)


def build_training_dataloader(tokenizer: PreTrainedTokenizerBase, config: TrainConfig) -> InfiniteDataLoader:
    dataset = OpenWebTextSequenceStream(
        tokenizer=tokenizer,
        sequence_length=config.total_tokens,
        shuffle_buffer=config.shuffle_buffer,
        seed=config.seed,
    )
    dataloader = DataLoader(dataset, batch_size=config.batch_size, num_workers=0)
    return InfiniteDataLoader(dataloader)


def split_prefix_and_suffix_for_exact_next_token_loss(
    input_ids: torch.Tensor,
    prefix_tokens: int,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    For exact next-token supervision on the suffix:
      - prefix_cache_ids: tokens [0, ..., prefix_tokens-2]
      - lm_input_ids:     tokens [prefix_tokens-1, ..., T-2]
      - lm_labels:        tokens [prefix_tokens,   ..., T-1]

    This lets the first suffix label be predicted from the last prefix token.
    """
    if prefix_tokens < 2:
        raise ValueError("prefix_tokens must be >= 2")
    prefix_cache_ids = input_ids[:, : prefix_tokens - 1]
    lm_input_ids = input_ids[:, prefix_tokens - 1 : -1]
    lm_labels = input_ids[:, prefix_tokens:]
    return prefix_cache_ids, lm_input_ids, lm_labels


class WarmupCosineScheduler:
    def __init__(self, optimizer: torch.optim.Optimizer, warmup_steps: int, total_steps: int) -> None:
        self.optimizer = optimizer
        self.warmup_steps = max(1, warmup_steps)
        self.total_steps = max(self.warmup_steps + 1, total_steps)
        self.step_id = 0
        self.base_lrs = [group["lr"] for group in optimizer.param_groups]

    def step(self) -> None:
        self.step_id += 1
        if self.step_id <= self.warmup_steps:
            multiplier = self.step_id / self.warmup_steps
        else:
            progress = (self.step_id - self.warmup_steps) / (self.total_steps - self.warmup_steps)
            multiplier = 0.5 * (1.0 + math.cos(math.pi * progress))
        for base_lr, param_group in zip(self.base_lrs, self.optimizer.param_groups):
            param_group["lr"] = base_lr * multiplier

    @property
    def lr(self) -> float:
        return self.optimizer.param_groups[0]["lr"]


def build_translator_pool(
    model_a: PreTrainedModel,
    model_b: PreTrainedModel,
    config: TrainConfig,
) -> Tuple[SharedKVTranslatorPool, Dict[str, ModelSpec]]:
    model_specs = {
        "A": get_model_spec(model_a),
        "B": get_model_spec(model_b),
    }
    translator_pool = SharedKVTranslatorPool(
        model_specs=model_specs,
        shared_slots=config.shared_slots,
        shared_dim=config.shared_dim,
        translator_dim=config.translator_dim,
        translator_heads=config.translator_heads,
        mlp_ratio=config.translator_mlp_ratio,
    )
    translator_pool.to(config.device)
    return translator_pool, model_specs


def build_models_and_tokenizer(config: TrainConfig) -> Tuple[PreTrainedModel, PreTrainedModel, PreTrainedTokenizerBase]:
    tokenizer = load_tokenizer(config.model_a_id)
    model_a = load_frozen_model(config.model_a_id, device=config.device, dtype=config.dtype)
    model_b = load_frozen_model(config.model_b_id, device=config.device, dtype=config.dtype)
    return model_a, model_b, tokenizer


def save_checkpoint(
    output_path: str,
    translator_pool: SharedKVTranslatorPool,
    optimizer: torch.optim.Optimizer,
    scheduler: WarmupCosineScheduler,
    train_config: TrainConfig,
    step: int,
    extra: Optional[Dict] = None,
) -> None:
    payload = {
        "translator_pool": translator_pool.state_dict(),
        "optimizer": optimizer.state_dict(),
        "step": step,
        "train_config": asdict(train_config),
        "scheduler_step": scheduler.step_id,
    }
    if extra is not None:
        payload["extra"] = extra
    output_path = str(output_path)
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    torch.save(payload, output_path)


def load_train_config_from_checkpoint(checkpoint_path: str) -> TrainConfig:
    payload = torch.load(checkpoint_path, map_location="cpu")
    return TrainConfig(**payload["train_config"])


def load_translator_pool_from_checkpoint(
    checkpoint_path: str,
    device_override: Optional[str] = None,
) -> Tuple[TrainConfig, SharedKVTranslatorPool, Dict[str, ModelSpec], PreTrainedModel, PreTrainedModel, PreTrainedTokenizerBase]:
    payload = torch.load(checkpoint_path, map_location="cpu")
    config = TrainConfig(**payload["train_config"])
    if device_override is not None:
        config.device = device_override
    model_a, model_b, tokenizer = build_models_and_tokenizer(config)
    translator_pool, model_specs = build_translator_pool(model_a, model_b, config)
    translator_pool.load_state_dict(payload["translator_pool"])
    translator_pool.to(config.device)
    translator_pool.eval()
    return config, translator_pool, model_specs, model_a, model_b, tokenizer


@torch.no_grad()
def prepare_generation_prefix(
    model: PreTrainedModel,
    tokenizer: PreTrainedTokenizerBase,
    text: str,
    device: str,
) -> Dict[str, torch.Tensor]:
    tokenized = tokenizer(text, return_tensors="pt")
    input_ids = tokenized.input_ids.to(device)
    if input_ids.shape[1] < 2:
        raise ValueError("Prefix text must tokenize to at least 2 tokens.")
    cache_ids = input_ids[:, :-1]
    seed_token = input_ids[:, -1:]
    past_key_values = extract_past_key_values(model, cache_ids)
    return {
        "full_prefix_ids": input_ids,
        "cache_ids": cache_ids,
        "seed_token": seed_token,
        "past_key_values": past_key_values,
    }


@torch.no_grad()
def generate_from_past(
    model: PreTrainedModel,
    seed_token: torch.Tensor,
    past_key_values: PastKeyValues,
    max_new_tokens: int,
    do_sample: bool = False,
    temperature: float = 1.0,
    top_k: int = 50,
    eos_token_id: Optional[int] = None,
) -> torch.Tensor:
    current_input = seed_token
    current_past = past_key_values
    generated = []
    for _ in range(max_new_tokens):
        outputs = model(input_ids=current_input, past_key_values=current_past, use_cache=True)
        logits = outputs.logits[:, -1, :]
        if temperature != 1.0:
            logits = logits / max(temperature, 1e-5)
        if do_sample:
            k = min(top_k, logits.shape[-1])
            values, indices = torch.topk(logits, k=k, dim=-1)
            probs = torch.softmax(values, dim=-1)
            sampled = torch.multinomial(probs, num_samples=1)
            next_token = indices.gather(-1, sampled)
        else:
            next_token = torch.argmax(logits, dim=-1, keepdim=True)
        generated.append(next_token)
        current_input = next_token
        current_past = outputs.past_key_values
        if eos_token_id is not None and torch.all(next_token.eq(eos_token_id)):
            break
    if not generated:
        return torch.empty(seed_token.shape[0], 0, dtype=seed_token.dtype, device=seed_token.device)
    return torch.cat(generated, dim=1)


def decode_full_generation(
    tokenizer: PreTrainedTokenizerBase,
    prefix_ids: torch.Tensor,
    generated_ids: torch.Tensor,
) -> str:
    full_ids = torch.cat([prefix_ids.cpu(), generated_ids.cpu()], dim=1)
    return tokenizer.decode(full_ids[0], skip_special_tokens=True)


def write_json(path: str, payload: Dict) -> None:
    path_obj = Path(path)
    path_obj.parent.mkdir(parents=True, exist_ok=True)
    with path_obj.open("w", encoding="utf-8") as fp:
        json.dump(payload, fp, indent=2, ensure_ascii=False)

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [ ]:
"""
Code 1
- Goal: train a toy cross-attention translator between GPT-2 and GPT-2 Medium
- Data: OpenWebText
- Objective: suffix LM loss only (bidirectional sum per step), no reconstruction loss

Important
- This example is intentionally compact and pragmatic.
- It follows the paper's high-level recipe with a shared latent space and per-model in/out translators,
  but uses a smaller, easier-to-read implementation suitable for experimentation.
- It is written to be compatible with transformers==4.35.2, where past_key_values are plain tuples,
  so DynamicCache is not used.
"""

from pathlib import Path

import torch
from tqdm.auto import tqdm



# -----------------------------------------------------------------------------
# No argparse by request. Edit values here.
# -----------------------------------------------------------------------------
CONFIG = TrainConfig(
    model_a_id="gpt2",
    model_b_id="gpt2-medium",
    output_dir="./outputs/lsc_toy",
    max_steps=100,
    batch_size=1,
    grad_accum_steps=16,
    total_tokens=128,
    prefix_tokens=64,
    learning_rate=1e-4,
    weight_decay=1e-2,
    warmup_steps=50,
    grad_clip_norm=1.0,
    log_every=25,
    save_every=500,
    seed=42,
    shuffle_buffer=50_000,
    shared_slots=32,
    shared_dim=128,
    translator_dim=256,
    translator_heads=4,
    translator_mlp_ratio=1,
    device="cuda" if torch.cuda.is_available() else "cpu",
    dtype="float32",
)


def main() -> None:
    set_seed(CONFIG.seed)
    output_dir = Path(CONFIG.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    write_json(output_dir / "train_config.json", CONFIG.__dict__)

    print(f"[Setup] device={CONFIG.device}")
    print(f"[Setup] loading models: A={CONFIG.model_a_id}, B={CONFIG.model_b_id}")
    model_a, model_b, tokenizer = build_models_and_tokenizer(CONFIG)
    translator_pool, model_specs = build_translator_pool(model_a, model_b, CONFIG)
    translator_pool.train()

    print("[Setup] model specs")
    print(f"  A: layers={model_specs['A'].num_layers}, hidden={model_specs['A'].hidden_size}, heads={model_specs['A'].num_heads}")
    print(f"  B: layers={model_specs['B'].num_layers}, hidden={model_specs['B'].hidden_size}, heads={model_specs['B'].num_heads}")
    print(f"[Setup] trainable translator params = {count_trainable_parameters(translator_pool):,}")

    dataloader = build_training_dataloader(tokenizer, CONFIG)

    optimizer = torch.optim.AdamW(
        translator_pool.parameters(),
        lr=CONFIG.learning_rate,
        weight_decay=CONFIG.weight_decay,
    )
    scheduler = WarmupCosineScheduler(
        optimizer=optimizer,
        warmup_steps=CONFIG.warmup_steps,
        total_steps=CONFIG.max_steps,
    )

    running_loss = 0.0
    progress_bar = tqdm(range(1, CONFIG.max_steps + 1), desc="Training")

    for step in progress_bar:
        optimizer.zero_grad(set_to_none=True)
        step_loss_value = 0.0

        for _ in range(CONFIG.grad_accum_steps):
            input_ids = next(dataloader).to(CONFIG.device)
            prefix_cache_ids, lm_input_ids, lm_labels = split_prefix_and_suffix_for_exact_next_token_loss(
                input_ids=input_ids,
                prefix_tokens=CONFIG.prefix_tokens,
            )

            with torch.no_grad():
                past_a = extract_past_key_values(model_a, prefix_cache_ids)
                past_b = extract_past_key_values(model_b, prefix_cache_ids)

            translated_a_to_b = translator_pool.translate_past_key_values(
                past_key_values=past_a,
                src_name="A",
                dst_name="B",
                dst_spec=model_specs["B"],
            )
            translated_b_to_a = translator_pool.translate_past_key_values(
                past_key_values=past_b,
                src_name="B",
                dst_name="A",
                dst_spec=model_specs["A"],
            )

            loss_a_to_b = compute_suffix_lm_loss(
                target_model=model_b,
                translated_past_key_values=translated_a_to_b,
                lm_input_ids=lm_input_ids,
                lm_labels=lm_labels,
            )
            loss_b_to_a = compute_suffix_lm_loss(
                target_model=model_a,
                translated_past_key_values=translated_b_to_a,
                lm_input_ids=lm_input_ids,
                lm_labels=lm_labels,
            )

            # Paper-style toy objective for this two-model setting:
            # sum of the two directional suffix-LM losses, no reconstruction loss.
            loss = loss_a_to_b + loss_b_to_a
            loss = loss / CONFIG.grad_accum_steps
            loss.backward()
            step_loss_value += loss.item()

        torch.nn.utils.clip_grad_norm_(translator_pool.parameters(), CONFIG.grad_clip_norm)
        optimizer.step()
        scheduler.step()

        running_loss += step_loss_value
        if step % CONFIG.log_every == 0:
            avg_loss = running_loss / CONFIG.log_every
            progress_bar.set_postfix(
                loss=f"{avg_loss:.4f}",
                lr=f"{scheduler.lr:.2e}",
            )
            print(
                f"[Step {step:04d}] total_bidirectional_suffix_lm_loss={avg_loss:.4f} | lr={scheduler.lr:.2e}"
            )
            running_loss = 0.0

        # if step % CONFIG.save_every == 0:
        #     checkpoint_path = output_dir / f"checkpoint_step_{step:04d}.pt"
        #     save_checkpoint(
        #         output_path=str(checkpoint_path),
        #         translator_pool=translator_pool,
        #         optimizer=optimizer,
        #         scheduler=scheduler,
        #         train_config=CONFIG,
        #         step=step,
        #         extra={
        #             "note": "Toy checkpoint trained with bidirectional suffix LM loss only.",
        #             "model_a": CONFIG.model_a_id,
        #             "model_b": CONFIG.model_b_id,
        #         },
        #     )
        #     print(f"[Checkpoint] saved to {checkpoint_path}")

    final_path = output_dir / "final_checkpoint.pt"
    save_checkpoint(
        output_path=str(final_path),
        translator_pool=translator_pool,
        optimizer=optimizer,
        scheduler=scheduler,
        train_config=CONFIG,
        step=CONFIG.max_steps,
        extra={
            "note": "Final toy checkpoint trained with bidirectional suffix LM loss only.",
            "model_a": CONFIG.model_a_id,
            "model_b": CONFIG.model_b_id,
        },
    )
    print(f"[Done] final checkpoint saved to {final_path}")


if __name__ == "__main__":
    main()

In [8]:
# eval.py
import logging
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, Optional

import torch
import torch.nn.functional as F
from datasets import load_dataset
from torch.utils.data import DataLoader, IterableDataset


@dataclass
class EvalConfig:
    checkpoint_path: str = "./outputs/lsc_toy/final_checkpoint.pt"
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    # evaluation sampling
    batch_size: int = 1
    num_workers: int = 0
    max_examples_per_dataset: int = 512
    seed: int = 42

    # streaming / shuffling
    shuffle_eval_stream: bool = True
    shuffle_buffer: int = 10_000

    # score sanity-check samples
    num_generation_samples_per_dataset: int = 2

    log_filename: str = "eval.log"


CONFIG = EvalConfig()


@dataclass
class HFDatasetSpec:
    name_for_log: str
    dataset_path: str
    dataset_name: Optional[str]
    split: str
    answer_mode: str
    question_field: str = "question"
    streaming: bool = False


class HFQAPairStream(IterableDataset):
    def __init__(
        self,
        spec: HFDatasetSpec,
        max_examples: int,
        shuffle: bool,
        seed: int,
        shuffle_buffer: int,
    ) -> None:
        super().__init__()
        self.spec = spec
        self.max_examples = max_examples
        self.shuffle = shuffle
        self.seed = seed
        self.shuffle_buffer = shuffle_buffer

    def _load_dataset(self):
        candidates = [(self.spec.dataset_path, self.spec.dataset_name)]

        last_error = None
        for dataset_path, dataset_name in candidates:
            try:
                if dataset_name is None:
                    return load_dataset(
                        dataset_path,
                        split=self.spec.split,
                        streaming=self.spec.streaming,
                    )
                return load_dataset(
                    dataset_path,
                    dataset_name,
                    split=self.spec.split,
                    streaming=self.spec.streaming,
                )
            except Exception as exc:
                last_error = exc

        raise RuntimeError(
            f"Failed to load dataset {self.spec.name_for_log} "
            f"with candidates={candidates}"
        ) from last_error

    def __iter__(self):
        dataset = self._load_dataset()
        if self.shuffle:
            if self.spec.streaming:
                dataset = dataset.shuffle(seed=self.seed, buffer_size=self.shuffle_buffer)
            else:
                dataset = dataset.shuffle(seed=self.seed)

        emitted = 0

        for example in dataset:
            qa_pair = extract_question_and_answer(self.spec, example)
            if qa_pair is None:
                continue

            yield qa_pair
            emitted += 1
            if emitted >= self.max_examples:
                return


class RunningAverage:
    def __init__(self) -> None:
        self.cosine_sum = 0.0
        self.accuracy_sum = 0.0
        self.native_accuracy_sum = 0.0
        self.count = 0

    def update(
        self,
        cosine_value: float,
        accuracy_value: float,
        native_accuracy_value: float,
        n: int,
    ) -> None:
        self.cosine_sum += float(cosine_value) * n
        self.accuracy_sum += float(accuracy_value) * n
        self.native_accuracy_sum += float(native_accuracy_value) * n
        self.count += n

    def summary(self) -> Dict[str, float]:
        if self.count == 0:
            return {
                "cosine": float("nan"),
                "accuracy": float("nan"),
                "native_accuracy": float("nan"),
                "count": 0,
            }
        return {
            "cosine": self.cosine_sum / self.count,
            "accuracy": self.accuracy_sum / self.count,
            "native_accuracy": self.native_accuracy_sum / self.count,
            "count": self.count,
        }


def setup_logger(log_path: Path) -> logging.Logger:
    log_path.parent.mkdir(parents=True, exist_ok=True)

    logger = logging.getLogger("lsc_eval")
    logger.setLevel(logging.INFO)
    logger.propagate = False

    for handler in list(logger.handlers):
        logger.removeHandler(handler)

    formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")

    file_handler = logging.FileHandler(log_path, encoding="utf-8")
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)

    stream_handler = logging.StreamHandler()
    stream_handler.setFormatter(formatter)
    logger.addHandler(stream_handler)

    return logger


def build_eval_dataloader(
    spec: HFDatasetSpec,
    eval_config: EvalConfig,
) -> DataLoader:
    dataset = HFQAPairStream(
        spec=spec,
        max_examples=eval_config.max_examples_per_dataset,
        shuffle=eval_config.shuffle_eval_stream,
        seed=eval_config.seed,
        shuffle_buffer=eval_config.shuffle_buffer,
    )
    return DataLoader(
        dataset,
        batch_size=eval_config.batch_size,
        num_workers=eval_config.num_workers,
        collate_fn=lambda batch: batch,
    )


def build_sample_dataloader(
    spec: HFDatasetSpec,
    eval_config: EvalConfig,
) -> DataLoader:
    dataset = HFQAPairStream(
        spec=spec,
        max_examples=eval_config.num_generation_samples_per_dataset,
        shuffle=False,
        seed=eval_config.seed,
        shuffle_buffer=eval_config.shuffle_buffer,
    )
    return DataLoader(
        dataset,
        batch_size=1,
        num_workers=eval_config.num_workers,
        collate_fn=lambda batch: batch,
    )


def extract_question_and_answer(spec: HFDatasetSpec, example: Dict) -> Optional[Dict[str, str]]:
    question = example.get(spec.question_field, "")
    if not isinstance(question, str) or not question.strip():
        return None

    if spec.answer_mode != "boolq":
        raise ValueError(f"Unsupported answer_mode: {spec.answer_mode}")

    answer_value = example.get("answer", None)
    if not isinstance(answer_value, bool):
        return None

    return {
        "question": question.strip(),
        "answer": "yes" if answer_value else "no",
    }


def format_question_prefix(question: str) -> str:
    return f"Question: {question.strip()}\nAnswer:"


def prepare_question_prefix(tokenizer, question: str, device: str) -> Dict[str, torch.Tensor]:
    prefix_text = format_question_prefix(question)
    tokenized = tokenizer(prefix_text, return_tensors="pt")
    input_ids = tokenized.input_ids.to(device)
    if input_ids.shape[1] < 2:
        raise ValueError("Question prefix must tokenize to at least 2 tokens.")
    cache_ids = input_ids[:, :-1]
    seed_token = input_ids[:, -1:]
    return {
        "prefix_text": prefix_text,
        "full_prefix_ids": input_ids,
        "cache_ids": cache_ids,
        "seed_token": seed_token,
    }


def build_boolq_choice_token_ids(tokenizer) -> Dict[str, torch.Tensor]:
    choice_token_ids = {}
    for label in ("yes", "no"):
        token_ids = tokenizer(f" {label}", add_special_tokens=False).input_ids
        if len(token_ids) < 1:
            raise ValueError(f"Failed to tokenize BoolQ label: {label}")
        choice_token_ids[label] = torch.tensor(token_ids, dtype=torch.long)
    return choice_token_ids


def score_candidate_logprob(
    model,
    past_key_values,
    seed_token: torch.Tensor,
    candidate_token_ids: torch.Tensor,
) -> float:
    device = seed_token.device
    candidate_ids = candidate_token_ids.to(device).unsqueeze(0)

    if candidate_ids.shape[1] == 1:
        scoring_input_ids = seed_token
    else:
        scoring_input_ids = torch.cat([seed_token, candidate_ids[:, :-1]], dim=1)

    outputs = model(
        input_ids=scoring_input_ids,
        past_key_values=past_key_values,
        use_cache=False,
    )
    log_probs = F.log_softmax(outputs.logits, dim=-1)
    token_log_probs = log_probs.gather(-1, candidate_ids.unsqueeze(-1)).squeeze(-1)
    return token_log_probs.sum(dim=1).item()


def score_boolq_choices(
    model,
    past_key_values,
    seed_token: torch.Tensor,
    boolq_choice_token_ids: Dict[str, torch.Tensor],
) -> Dict[str, float]:
    return {
        "yes": score_candidate_logprob(
            model=model,
            past_key_values=past_key_values,
            seed_token=seed_token,
            candidate_token_ids=boolq_choice_token_ids["yes"],
        ),
        "no": score_candidate_logprob(
            model=model,
            past_key_values=past_key_values,
            seed_token=seed_token,
            candidate_token_ids=boolq_choice_token_ids["no"],
        ),
    }


def predict_boolq_label(choice_scores: Dict[str, float]) -> str:
    return "yes" if choice_scores["yes"] >= choice_scores["no"] else "no"


def summarize_path_metrics(path_metrics: Dict[str, RunningAverage]) -> Dict[str, Dict[str, float]]:
    results = {}
    total_cosine_sum = 0.0
    total_accuracy_sum = 0.0
    total_native_accuracy_sum = 0.0
    total_count = 0

    for path_name, meter in path_metrics.items():
        path_result = meter.summary()
        results[path_name] = path_result

        total_cosine_sum += meter.cosine_sum
        total_accuracy_sum += meter.accuracy_sum
        total_native_accuracy_sum += meter.native_accuracy_sum
        total_count += meter.count

    if total_count == 0:
        results["AVG"] = {
            "cosine": float("nan"),
            "accuracy": float("nan"),
            "native_accuracy": float("nan"),
            "count": 0,
        }
    else:
        results["AVG"] = {
            "cosine": total_cosine_sum / total_count,
            "accuracy": total_accuracy_sum / total_count,
            "native_accuracy": total_native_accuracy_sum / total_count,
            "count": total_count,
        }

    return results


def summarize_overall_results(
    all_results: Dict[str, Dict[str, Dict[str, float]]]
) -> Dict[str, Dict[str, float]]:
    overall = {
        "A_to_B": {"cosine_sum": 0.0, "accuracy_sum": 0.0, "native_accuracy_sum": 0.0, "count": 0},
        "B_to_A": {"cosine_sum": 0.0, "accuracy_sum": 0.0, "native_accuracy_sum": 0.0, "count": 0},
    }

    for dataset_result in all_results.values():
        for key in ("A_to_B", "B_to_A"):
            row = dataset_result[key]
            count = int(row["count"])
            overall[key]["cosine_sum"] += float(row["cosine"]) * count
            overall[key]["accuracy_sum"] += float(row["accuracy"]) * count
            overall[key]["native_accuracy_sum"] += float(row["native_accuracy"]) * count
            overall[key]["count"] += count

    summarized = {}
    total_cosine_sum = 0.0
    total_accuracy_sum = 0.0
    total_native_accuracy_sum = 0.0
    total_count = 0

    for key in ("A_to_B", "B_to_A"):
        count = overall[key]["count"]
        if count == 0:
            summarized[key] = {
                "cosine": float("nan"),
                "accuracy": float("nan"),
                "native_accuracy": float("nan"),
                "count": 0,
            }
        else:
            summarized[key] = {
                "cosine": overall[key]["cosine_sum"] / count,
                "accuracy": overall[key]["accuracy_sum"] / count,
                "native_accuracy": overall[key]["native_accuracy_sum"] / count,
                "count": count,
            }

        total_cosine_sum += overall[key]["cosine_sum"]
        total_accuracy_sum += overall[key]["accuracy_sum"]
        total_native_accuracy_sum += overall[key]["native_accuracy_sum"]
        total_count += count

    if total_count == 0:
        summarized["AVG"] = {
            "cosine": float("nan"),
            "accuracy": float("nan"),
            "native_accuracy": float("nan"),
            "count": 0,
        }
    else:
        summarized["AVG"] = {
            "cosine": total_cosine_sum / total_count,
            "accuracy": total_accuracy_sum / total_count,
            "native_accuracy": total_native_accuracy_sum / total_count,
            "count": total_count,
        }

    return summarized


@torch.inference_mode()
def evaluate_dataset(
    spec: HFDatasetSpec,
    dataloader: DataLoader,
    tokenizer,
    train_config,
    translator_pool,
    model_specs,
    model_a,
    model_b,
    logger: logging.Logger,
) -> Dict[str, Dict[str, float]]:
    device = train_config.device
    boolq_choice_token_ids = build_boolq_choice_token_ids(tokenizer)

    path_metrics = {
        "A_to_B": RunningAverage(),
        "B_to_A": RunningAverage(),
    }

    processed_examples = 0

    for batch_idx, batch in enumerate(dataloader, start=1):
        for example in batch:
            question = example["question"]
            gold_answer = example["answer"]

            prefix = prepare_question_prefix(tokenizer=tokenizer, question=question, device=device)
            prefix_cache_ids = prefix["cache_ids"]
            seed_token = prefix["seed_token"]

            past_a = extract_past_key_values(model_a, prefix_cache_ids)
            past_b = extract_past_key_values(model_b, prefix_cache_ids)

            translated_a_to_b = translator_pool.translate_past_key_values(
                past_key_values=past_a,
                src_name="A",
                dst_name="B",
                dst_spec=model_specs["B"],
            )
            translated_b_to_a = translator_pool.translate_past_key_values(
                past_key_values=past_b,
                src_name="B",
                dst_name="A",
                dst_spec=model_specs["A"],
            )

            cosine_a_to_b = cosine_similarity_between_past(translated_a_to_b, past_b)
            cosine_b_to_a = cosine_similarity_between_past(translated_b_to_a, past_a)

            translated_scores_b = score_boolq_choices(
                model=model_b,
                past_key_values=translated_a_to_b,
                seed_token=seed_token,
                boolq_choice_token_ids=boolq_choice_token_ids,
            )
            translated_scores_a = score_boolq_choices(
                model=model_a,
                past_key_values=translated_b_to_a,
                seed_token=seed_token,
                boolq_choice_token_ids=boolq_choice_token_ids,
            )

            native_scores_b = score_boolq_choices(
                model=model_b,
                past_key_values=past_b,
                seed_token=seed_token,
                boolq_choice_token_ids=boolq_choice_token_ids,
            )
            native_scores_a = score_boolq_choices(
                model=model_a,
                past_key_values=past_a,
                seed_token=seed_token,
                boolq_choice_token_ids=boolq_choice_token_ids,
            )

            translated_pred_b = predict_boolq_label(translated_scores_b)
            translated_pred_a = predict_boolq_label(translated_scores_a)
            native_pred_b = predict_boolq_label(native_scores_b)
            native_pred_a = predict_boolq_label(native_scores_a)

            acc_a_to_b = 1.0 if translated_pred_b == gold_answer else 0.0
            acc_b_to_a = 1.0 if translated_pred_a == gold_answer else 0.0
            native_acc_b = 1.0 if native_pred_b == gold_answer else 0.0
            native_acc_a = 1.0 if native_pred_a == gold_answer else 0.0

            path_metrics["A_to_B"].update(cosine_a_to_b, acc_a_to_b, native_acc_b, 1)
            path_metrics["B_to_A"].update(cosine_b_to_a, acc_b_to_a, native_acc_a, 1)

            processed_examples += 1

        if batch_idx % 50 == 0:
            logger.info(
                "[%s] progress: %d/%d examples",
                spec.name_for_log,
                processed_examples,
                CONFIG.max_examples_per_dataset,
            )

    return summarize_path_metrics(path_metrics)


def log_dataset_result(
    logger: logging.Logger,
    dataset_name: str,
    results: Dict[str, Dict[str, float]],
    model_a_id: str,
    model_b_id: str,
) -> None:
    pretty_names = {
        "A_to_B": f"A_to_B ({model_a_id} -> {model_b_id})",
        "B_to_A": f"B_to_A ({model_b_id} -> {model_a_id})",
        "AVG": "AVG (weighted over both paths)",
    }

    logger.info("===== %s =====", dataset_name)
    for key in ("A_to_B", "B_to_A", "AVG"):
        row = results[key]
        logger.info(
            "%s | cosine=%.6f | accuracy=%.6f | native_accuracy=%.6f | count=%d",
            pretty_names[key],
            row["cosine"],
            row["accuracy"],
            row["native_accuracy"],
            int(row["count"]),
        )


def log_overall_result(
    logger: logging.Logger,
    results: Dict[str, Dict[str, float]],
    model_a_id: str,
    model_b_id: str,
) -> None:
    pretty_names = {
        "A_to_B": f"A_to_B ({model_a_id} -> {model_b_id})",
        "B_to_A": f"B_to_A ({model_b_id} -> {model_a_id})",
        "AVG": "AVG (weighted over all datasets and both paths)",
    }

    logger.info("===== OVERALL AVERAGE ACROSS DATASETS =====")
    for key in ("A_to_B", "B_to_A", "AVG"):
        row = results[key]
        logger.info(
            "%s | cosine=%.6f | accuracy=%.6f | native_accuracy=%.6f | count=%d",
            pretty_names[key],
            row["cosine"],
            row["accuracy"],
            row["native_accuracy"],
            int(row["count"]),
        )


@torch.inference_mode()
def log_boolq_score_samples(
    spec: HFDatasetSpec,
    tokenizer,
    train_config,
    eval_config: EvalConfig,
    translator_pool,
    model_specs,
    model_a,
    model_b,
    logger: logging.Logger,
) -> None:
    if eval_config.num_generation_samples_per_dataset <= 0:
        return

    sample_dataloader = build_sample_dataloader(
        spec=spec,
        eval_config=eval_config,
    )
    boolq_choice_token_ids = build_boolq_choice_token_ids(tokenizer)

    logger.info("===== SAMPLE BOOLQ SCORES: %s =====", spec.name_for_log)

    sample_idx = 0
    for batch in sample_dataloader:
        example = batch[0]
        question = example["question"]
        gold_answer = example["answer"]

        prefix = prepare_question_prefix(tokenizer=tokenizer, question=question, device=train_config.device)
        prefix_cache_ids = prefix["cache_ids"]
        seed_token = prefix["seed_token"]
        prefix_text = prefix["prefix_text"]

        past_a = extract_past_key_values(model_a, prefix_cache_ids)
        past_b = extract_past_key_values(model_b, prefix_cache_ids)

        translated_a_to_b = translator_pool.translate_past_key_values(
            past_key_values=past_a,
            src_name="A",
            dst_name="B",
            dst_spec=model_specs["B"],
        )
        translated_b_to_a = translator_pool.translate_past_key_values(
            past_key_values=past_b,
            src_name="B",
            dst_name="A",
            dst_spec=model_specs["A"],
        )

        translated_scores_b = score_boolq_choices(
            model=model_b,
            past_key_values=translated_a_to_b,
            seed_token=seed_token,
            boolq_choice_token_ids=boolq_choice_token_ids,
        )
        translated_scores_a = score_boolq_choices(
            model=model_a,
            past_key_values=translated_b_to_a,
            seed_token=seed_token,
            boolq_choice_token_ids=boolq_choice_token_ids,
        )

        native_scores_b = score_boolq_choices(
            model=model_b,
            past_key_values=past_b,
            seed_token=seed_token,
            boolq_choice_token_ids=boolq_choice_token_ids,
        )
        native_scores_a = score_boolq_choices(
            model=model_a,
            past_key_values=past_a,
            seed_token=seed_token,
            boolq_choice_token_ids=boolq_choice_token_ids,
        )

        translated_pred_b = predict_boolq_label(translated_scores_b)
        translated_pred_a = predict_boolq_label(translated_scores_a)
        native_pred_b = predict_boolq_label(native_scores_b)
        native_pred_a = predict_boolq_label(native_scores_a)

        logger.info(
            "--- Sample %d/%d | %s ---",
            sample_idx + 1,
            eval_config.num_generation_samples_per_dataset,
            spec.name_for_log,
        )
        logger.info("prefix(question):\n%s", prefix_text)
        logger.info("gold_answer: %s", gold_answer)
        logger.info(
            "A_to_B translated_scores: yes=%.6f | no=%.6f | pred=%s | correct=%s",
            translated_scores_b["yes"],
            translated_scores_b["no"],
            translated_pred_b,
            translated_pred_b == gold_answer,
        )
        logger.info(
            "A_to_B native_baseline_scores: yes=%.6f | no=%.6f | pred=%s | correct=%s",
            native_scores_b["yes"],
            native_scores_b["no"],
            native_pred_b,
            native_pred_b == gold_answer,
        )
        logger.info(
            "B_to_A translated_scores: yes=%.6f | no=%.6f | pred=%s | correct=%s",
            translated_scores_a["yes"],
            translated_scores_a["no"],
            translated_pred_a,
            translated_pred_a == gold_answer,
        )
        logger.info(
            "B_to_A native_baseline_scores: yes=%.6f | no=%.6f | pred=%s | correct=%s",
            native_scores_a["yes"],
            native_scores_a["no"],
            native_pred_a,
            native_pred_a == gold_answer,
        )

        sample_idx += 1
        if sample_idx >= eval_config.num_generation_samples_per_dataset:
            break


def main() -> None:
    eval_config = CONFIG
    set_seed(eval_config.seed)

    checkpoint_path = Path(eval_config.checkpoint_path)
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

    log_path = checkpoint_path.parent / eval_config.log_filename
    logger = setup_logger(log_path)
    logger.info("Starting evaluation")
    logger.info("checkpoint_path=%s", checkpoint_path)
    logger.info("eval_config=%s", asdict(eval_config))

    train_config, translator_pool, model_specs, model_a, model_b, tokenizer = load_translator_pool_from_checkpoint(
        checkpoint_path=str(checkpoint_path),
        device_override=eval_config.device,
    )

    translator_pool.eval()
    model_a.eval()
    model_b.eval()

    logger.info("restored_train_config=%s", asdict(train_config))
    logger.info("model_A=%s", train_config.model_a_id)
    logger.info("model_B=%s", train_config.model_b_id)
    logger.info("translation_mode=full_past_key_values")
    logger.info("qa_eval_log_path=%s", log_path)

    dataset_specs = [
        HFDatasetSpec(
            name_for_log="BoolQ/validation",
            dataset_path="boolq",
            dataset_name=None,
            split="validation",
            answer_mode="boolq",
            question_field="question",
            streaming=False,
        ),
    ]

    all_results = {}

    for spec in dataset_specs:
        logger.info("Preparing dataloader for %s", spec.name_for_log)
        dataloader = build_eval_dataloader(
            spec=spec,
            eval_config=eval_config,
        )

        results = evaluate_dataset(
            spec=spec,
            dataloader=dataloader,
            tokenizer=tokenizer,
            train_config=train_config,
            translator_pool=translator_pool,
            model_specs=model_specs,
            model_a=model_a,
            model_b=model_b,
            logger=logger,
        )
        all_results[spec.name_for_log] = results

        log_dataset_result(
            logger=logger,
            dataset_name=spec.name_for_log,
            results=results,
            model_a_id=train_config.model_a_id,
            model_b_id=train_config.model_b_id,
        )

        log_boolq_score_samples(
            spec=spec,
            tokenizer=tokenizer,
            train_config=train_config,
            eval_config=eval_config,
            translator_pool=translator_pool,
            model_specs=model_specs,
            model_a=model_a,
            model_b=model_b,
            logger=logger,
        )

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    logger.info("===== FINAL SUMMARY =====")
    for dataset_name, result in all_results.items():
        logger.info(
            "%s | A_to_B(cos=%.6f, acc=%.6f, native_acc=%.6f) | "
            "B_to_A(cos=%.6f, acc=%.6f, native_acc=%.6f) | "
            "AVG(cos=%.6f, acc=%.6f, native_acc=%.6f)",
            dataset_name,
            result["A_to_B"]["cosine"],
            result["A_to_B"]["accuracy"],
            result["A_to_B"]["native_accuracy"],
            result["B_to_A"]["cosine"],
            result["B_to_A"]["accuracy"],
            result["B_to_A"]["native_accuracy"],
            result["AVG"]["cosine"],
            result["AVG"]["accuracy"],
            result["AVG"]["native_accuracy"],
        )

    overall_results = summarize_overall_results(all_results)
    log_overall_result(
        logger=logger,
        results=overall_results,
        model_a_id=train_config.model_a_id,
        model_b_id=train_config.model_b_id,
    )

    logger.info("Done. Saved log to %s", log_path)


if __name__ == "__main__":
    main()

2026-03-09 00:01:47,026 | INFO | Starting evaluation
2026-03-09 00:01:47,026 | INFO | checkpoint_path=outputs/lsc_toy/final_checkpoint.pt
2026-03-09 00:01:47,027 | INFO | eval_config={'checkpoint_path': './outputs/lsc_toy/final_checkpoint.pt', 'device': 'cuda', 'batch_size': 1, 'num_workers': 0, 'max_examples_per_dataset': 512, 'seed': 42, 'shuffle_eval_stream': True, 'shuffle_buffer': 10000, 'num_generation_samples_per_dataset': 2, 'log_filename': 'eval.log'}
2026-03-09 00:02:02,376 | INFO | restored_train_config={'model_a_id': 'gpt2', 'model_b_id': 'gpt2-medium', 'output_dir': './outputs/lsc_toy', 'max_steps': 100, 'batch_size': 1, 'grad_accum_steps': 16, 'total_tokens': 128, 'prefix_tokens': 64, 'learning_rate': 0.0001, 'weight_decay': 0.01, 'warmup_steps': 50, 'grad_clip_norm': 1.0, 'log_every': 25, 'save_every': 100, 'seed': 42, 'shuffle_buffer': 10000, 'shared_slots': 16, 'shared_dim': 128, 'translator_dim': 256, 'translator_heads': 8, 'translator_mlp_ratio': 2, 'device': 'cuda',

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.69M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/1.26M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9427 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3270 [00:00<?, ? examples/s]

2026-03-09 00:02:16,591 | INFO | [BoolQ/validation] progress: 50/512 examples
2026-03-09 00:02:27,717 | INFO | [BoolQ/validation] progress: 100/512 examples
2026-03-09 00:02:38,711 | INFO | [BoolQ/validation] progress: 150/512 examples
2026-03-09 00:02:49,768 | INFO | [BoolQ/validation] progress: 200/512 examples
2026-03-09 00:03:00,772 | INFO | [BoolQ/validation] progress: 250/512 examples
2026-03-09 00:03:11,859 | INFO | [BoolQ/validation] progress: 300/512 examples
2026-03-09 00:03:22,890 | INFO | [BoolQ/validation] progress: 350/512 examples
2026-03-09 00:03:33,940 | INFO | [BoolQ/validation] progress: 400/512 examples
2026-03-09 00:03:44,987 | INFO | [BoolQ/validation] progress: 450/512 examples
2026-03-09 00:03:56,055 | INFO | [BoolQ/validation] progress: 500/512 examples
2026-03-09 00:03:58,673 | INFO | ===== BoolQ/validation =====
2026-03-09 00:03:58,674 | INFO | A_to_B (gpt2 -> gpt2-medium) | cosine=0.078311 | accuracy=0.425781 | native_accuracy=0.558594 | count=512
2026-03-09